# 02. Contrastive Fine-Tuning (`TextExperimentAdapter` + `ExperimentTracker`)

In [1]:
from pathlib import Path

import mlflow
import pandas as pd

from meme_organizer.config import MLFLOW_EXPERIMENT_FUSION, MLFLOW_EXPERIMENT_TEXT
from meme_organizer.evaluation.metrics.experiment_tracker import ExperimentTracker
from meme_organizer.evaluation.pipeline.evaluation_dataset_preparer import EvaluationDatasetPreparer
from meme_organizer.retrieval.fusion.hybrid_retriever import HybridRetriever
from meme_organizer.retrieval.text.bm25_retriever import BM25Retriever
from meme_organizer.retrieval.text.dense_text_retriever import DenseTextRetriever
from meme_organizer.retrieval.text.fine_tuned_dense_retriever import FineTunedDenseRetriever
from meme_organizer.retrieval.text.text_experiment_adapter import TextExperimentAdapter

ROOT = Path("..").resolve()
split_data = EvaluationDatasetPreparer().prepare(
    ROOT / "data" / "gold" / "gold_dataset.jsonl",
    ROOT / "data" / "splits" / "splits_v1.json",
    "test",
)
print(f"Corpus: {len(split_data.corpus_docs)} | Test queries: {len(split_data.queries)}")

2026/11/27 16:36:05 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Corpus: 318 | Test queries: 153


In [2]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_TEXT) as exp:
    res_rubert_ft = exp.create_pipeline(
        TextExperimentAdapter(
            "rubert-tiny2_lora",
            FineTunedDenseRetriever("rubert-tiny2-finetuned", "cointegrated/rubert-tiny2"),
            split_data,
            train_mode=True,
        )
    ).run()
print("RuBERT LoRA:", res_rubert_ft["quality"])

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run rubert-tiny2_lora at: http://localhost:5000/#/experiments/12/runs/5b1b7f174f794af9aed04431a40a698d
🧪 View experiment at: http://localhost:5000/#/experiments/12
RuBERT LoRA: {'recall_at_1': 0.4641, 'recall_at_5': 0.6078, 'mrr_at_5': 0.5154, 'ndcg_at_5': 0.5383, 'total_queries': 153}


In [3]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_TEXT) as exp:
    res_e5_ft = exp.create_pipeline(
        TextExperimentAdapter(
            "e5-small_lora",
            FineTunedDenseRetriever("e5-small-finetuned", "intfloat/multilingual-e5-small"),
            split_data,
            train_mode=True,
        )
    ).run()
print("E5 LoRA:", res_e5_ft["quality"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run e5-small_lora at: http://localhost:5000/#/experiments/12/runs/3bc23e4fb1f4443486fecd5ad8e837bc
🧪 View experiment at: http://localhost:5000/#/experiments/12
E5 LoRA: {'recall_at_1': 0.5621, 'recall_at_5': 0.7124, 'mrr_at_5': 0.6263, 'ndcg_at_5': 0.6482, 'total_queries': 153}
